# Automated Data Labeling System
This project provides a tool to upload datasets and automatically generate descriptive labels for each row using local natural language processing models.

In [ ]:
import pandas as pd
import os
from google.colab import files
from transformers import pipeline

# 1. Setup Output Directory
output_dir = 'Output'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print("Please upload your Excel or CSV file:")
uploaded = files.upload()

In [ ]:
from transformers import pipeline
import torch

# Load a local advanced Zero-Shot Classification model
# This runs entirely on the Colab CPU/GPU without any API calls
print("Loading local advanced model (Zero-Shot Classification)...")
device = 0 if torch.cuda.is_available() else -1
classifier = pipeline("zero-shot-classification", model="valhalla/distilbart-mnli-12-3", device=device)

In [ ]:
# 3. Process the file using Local Model with Dynamic Candidate Generation
import re
import io

for filename, content in uploaded.items():
    # Fix: Use BytesIO to read directly from uploaded memory content
    if filename.endswith('.csv'):
        df = pd.read_csv(io.BytesIO(content))
    elif filename.endswith(('.xls', '.xlsx')):
        df = pd.read_excel(io.BytesIO(content))
    else:
        continue

    print(f"Processing {len(df)} rows from {filename} using local model...")

    def generate_local_label(row):
        # 1. Prepare row context
        row_items = [str(val) for val in row.values if pd.notnull(val) and str(val).strip() != ""]
        row_text = " ".join(row_items)

        # 2. Dynamic Labeling Technique: 
        # We extract distinct words from the row itself to use as 'dynamic candidates'
        words = re.findall(r'\b[A-Z][a-z]{3,}\b', row_text)
        candidates = list(set(words))[:5] 
        if len(candidates) < 2:
            candidates += ["General", "Information", "Data", "Category"]

        try:
            # Local inference
            result = classifier(row_text[:512], candidates)
            return result['labels'][0]
        except:
            return "Miscellaneous"

    # Apply the local labeling process
    df['DataLabel'] = df.apply(generate_local_label, axis=1)

    # 4. Save Output
    base_name = os.path.splitext(filename)[0]
    output_filename = f"{base_name}DatalabelledReport.csv"
    output_path = os.path.join(output_dir, output_filename)
    
    df.to_csv(output_path, index=False)
    print(f"Successfully saved to: {output_path}")
    display(df.head())